In [ ]:
import json
import torch
from datasets import Dataset
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
model_name = "microsoft/Phi-3.5-mini-instruct"
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map=device)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

In [ ]:
data = json.load(open('instruction-data.json'))

In [ ]:
print(f"{sum(p.nelement() for p in model.parameters()):,}")

3,821,079,552


In [ ]:
model.get_memory_footprint() / (1024**3)

7.117315649986267

In [ ]:
prompt = "Who are you?"

tokenizer.apply_chat_template([
    {'role': 'user', 'content': prompt},
    {'role': 'assistant', 'content': 'I am good'}
], tokenize=False)

'<|user|>\nWho are you?<|end|>\n<|assistant|>\nI am good<|end|>\n<|endoftext|>'

In [ ]:
# <|user|>
# Who are you?<|end|>
# <|assistant|>
# I am good<|end|>
# <|endoftext|>

In [ ]:
data[0]

{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
 'input': 'freind --> friend',
 'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}

In [ ]:
def apply_chat_templete(example):
    user_text = example['instruction'] + "\n" + example['input']
    ai_text = example['output']

    tokenized = tokenizer.apply_chat_template([
        {'role': 'user', 'content': user_text},
        {'role': 'assistant', 'content': ai_text}
    ], tokenize=False)

    encoded = tokenizer(tokenized, truncation=True, padding=True)


    return {
        'input_ids': encoded['input_ids'],
        'attention_mask': encoded['attention_mask'],
        'labels': encoded['input_ids'].copy() # I should fix that place
    }

# print(apply_chat_templete(data[0]))

In [ ]:
tokenized_text = []

for example in data:
    tokenized = apply_chat_templete(example)
    tokenized_text.append(tokenized)

In [ ]:
# train_ds =Dataset.from_dict({'text': tokenized_text})
raw_ds = Dataset.from_list(data)

In [ ]:
train_ds = raw_ds.map(apply_chat_templete, remove_columns=raw_ds.column_names)

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

In [ ]:
train_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1100
})

In [ ]:
args = TrainingArguments(
    output_dir = f"{model_name}-finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    optim='adamw_torch',
    report_to='none'
)

In [ ]:
trainer =Trainer(
    model = model,
    processing_class=tokenizer,
    train_dataset = train_ds,
    data_collator=data_collator
)

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [ ]:
trainer.train()

OutOfMemoryError: CUDA out of memory. Tried to allocate 48.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 30.12 MiB is free. Process 4498 has 14.71 GiB memory in use. Of the allocated memory 14.15 GiB is allocated by PyTorch, and 440.43 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
model.model.embed_tokens.weight.shape

torch.Size([32064, 3072])